# Lab 1: VPG & PPO in PyTorch

**Course:** Deep Reinforcement Learning · **Framework:** PyTorch · **Estimated time:** ~60 min

## Learning Objectives

- Implement a minimal Vanilla Policy Gradient training loop from scratch and verify it learns CartPole-v1 within 50 epochs
- Extend VPG to PPO-Clip: clipped surrogate objective, multiple gradient steps, KL early stopping
- Run Spinning Up's full PPO on LunarLander-v2 and compare λ, network size, and VPG baseline
- Design and execute a systematic ExperimentGrid sweep across 54 PPO configurations
- Implement the diagonal Gaussian log-likelihood function (Problem Set 1.1)
- Implement an MLP diagonal Gaussian policy for PPO (Problem Set 1.2)
- Implement the TD3 critic and policy loss functions (Problem Set 1.3)

## Setup

Install dependencies and clone Spinning Up. We patch `spinup/__init__.py` to skip TF1 imports, which fail on modern Python but are not needed for the PyTorch implementations.

In [ ]:
!pip install mpi4py gymnasium -q
!git clone https://github.com/openai/spinningup.git 2>/dev/null || echo "Already cloned"
%cd spinningup
!pip install -e . -q

In [ ]:
# Patch spinup/__init__.py to skip TF1 imports (incompatible with Python 3.10+)
init_path = 'spinup/__init__.py'
with open(init_path) as f:
    lines = f.readlines()

patched = []
for line in lines:
    if 'tf1' in line and line.strip().startswith('from'):
        patched.append('# ' + line)  # comment out TF1 imports
    else:
        patched.append(line)

with open(init_path, 'w') as f:
    f.writelines(patched)

print("Patched. Verifying PyTorch VPG loads:")
from spinup.algos.pytorch.vpg import vpg
print("OK")

In [ ]:
# Verify: should print epoch-by-epoch AverageEpRet logs
!python -m spinup.run vpg_pytorch --env CartPole-v1 --epochs 5

## Exercise 1: Minimal VPG from Scratch

Implement a minimal VPG training loop using CartPole-v1. **Do not use Spinning Up's VPG implementation** — use it only as a reference.

Your implementation must:
1. Build a categorical policy network (discrete action space)
2. Collect one epoch of trajectories by rolling out the policy
3. Compute rewards-to-go for each timestep
4. Compute the policy gradient loss and take one gradient step

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gym  # Spinning Up uses gym; we match it here for consistency

class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),     nn.Tanh(),
            nn.Linear(64, act_dim)
        )

    def forward(self, obs):
        return Categorical(logits=self.net(obs))


def rewards_to_go(rewards):
    n = len(rewards)
    rtg = torch.zeros(n)
    running = 0
    for t in reversed(range(n)):
        running = rewards[t] + running
        rtg[t] = running
    return rtg


def gym_reset(env):
    '''Handle both old gym (returns obs) and new gym (returns obs, info).'''
    result = env.reset()
    return result[0] if isinstance(result, tuple) else result


def gym_step(env, action):
    '''Handle both old gym (obs,rew,done,info) and new gym (obs,rew,term,trunc,info).'''
    result = env.step(action)
    obs, rew = result[0], result[1]
    done = result[2] if len(result) == 4 else (result[2] or result[3])
    return obs, rew, done


def collect_epoch(env, policy, steps=4000):
    obs_buf, act_buf, rtg_buf = [], [], []
    obs = gym_reset(env)
    ep_rewards = []

    for _ in range(steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        dist  = policy(obs_t)
        act   = dist.sample()

        obs_buf.append(obs)
        act_buf.append(act.item())

        obs, rew, done = gym_step(env, act.item())
        ep_rewards.append(rew)

        if done:
            rtg_buf.extend(rewards_to_go(ep_rewards).tolist())
            obs = gym_reset(env)
            ep_rewards = []

    return (
        torch.as_tensor(obs_buf, dtype=torch.float32),
        torch.as_tensor(act_buf, dtype=torch.int32),
        torch.as_tensor(rtg_buf, dtype=torch.float32),
    )


def train_vpg(env_name='CartPole-v1', epochs=50, steps=4000, lr=3e-4):
    env    = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optim  = torch.optim.Adam(policy.parameters(), lr=lr)

    for epoch in range(epochs):
        obs, acts, rtg = collect_epoch(env, policy, steps)
        optim.zero_grad()
        loss = -(policy(obs).log_prob(acts) * rtg).mean()
        loss.backward()
        optim.step()
        print(f'Epoch {epoch+1:3d} | mean_rtg={rtg.mean():.1f} | loss={loss.item():.4f}')

    return policy

policy = train_vpg()

**Tasks — try these modifications:**
- Add a value function network V(s) and use RTG - V(s) as the advantage weights. Does it converge faster?
- Plot the mean RTG per epoch to visualise learning progress.

## Exercise 2: Implement PPO-Clip

Extend VPG to PPO by adding:
1. The clipped surrogate objective
2. Multiple gradient steps per epoch (`train_pi_iters`)
3. Approximate KL early stopping

In [ ]:
class ValueNet(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),     nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


def compute_ppo_loss(obs, acts, adv, logp_old, policy, clip_ratio=0.2):
    dist  = policy(obs)
    logp  = dist.log_prob(acts)
    ratio = torch.exp(logp - logp_old)

    clipped   = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio)
    loss      = -torch.min(ratio * adv, clipped * adv).mean()
    approx_kl = (logp_old - logp).mean().item()
    return loss, approx_kl


def train_ppo(env_name='CartPole-v1', epochs=50, steps=4000,
              pi_lr=3e-4, v_lr=1e-3, clip_ratio=0.2, target_kl=0.01):
    env    = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    vf     = ValueNet(env.observation_space.shape[0])
    pi_opt = torch.optim.Adam(policy.parameters(), lr=pi_lr)
    v_opt  = torch.optim.Adam(vf.parameters(), lr=v_lr)

    for epoch in range(epochs):
        obs, acts, rtg = collect_epoch(env, policy, steps)

        with torch.no_grad():
            vals     = vf(obs)
            adv      = rtg - vals
            adv      = (adv - adv.mean()) / (adv.std() + 1e-8)
            logp_old = policy(obs).log_prob(acts)

        # Policy update with KL early stopping
        for i in range(80):
            pi_opt.zero_grad()
            loss, kl = compute_ppo_loss(obs, acts, adv, logp_old, policy, clip_ratio)
            if kl > 1.5 * target_kl:
                print(f'  Epoch {epoch+1}: KL early stop at step {i}, KL={kl:.4f}')
                break
            loss.backward()
            pi_opt.step()

        # Value function update
        for _ in range(80):
            v_opt.zero_grad()
            v_loss = ((vf(obs) - rtg)**2).mean()
            v_loss.backward()
            v_opt.step()

        print(f'Epoch {epoch+1:3d} | mean_rtg={rtg.mean():.1f} | pi_loss={loss.item():.4f} | v_loss={v_loss.item():.4f}')

train_ppo()

**Tasks:**
- Vary `clip_ratio` across `[0.1, 0.2, 0.3, 0.5]`. How does it affect convergence speed and stability?
- What happens at `clip_ratio=0.01` (too restrictive) vs `clip_ratio=0.5` (too permissive)?

## Exercise 3: Spinning Up's PPO on LunarLander-v2

Use Spinning Up's full PPO (with GAE-Lambda, structured logging, and proper value fitting) on a harder environment. Each command runs 3 seeds automatically.

In [ ]:
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[64,64]" \
    --epochs 150 \
    --lam 0.97 \
    --clip_ratio 0.2 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-baseline

In [ ]:
# Lower lambda: more bias, less variance
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[64,64]" \
    --epochs 150 \
    --lam 0.9 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-lam09

In [ ]:
# Smaller network
!python -m spinup.run ppo_pytorch \
    --env LunarLander-v2 \
    --hid "[32,32]" \
    --epochs 150 \
    --seed 0 10 20 \
    --exp_name ppo-lunar-small

In [ ]:
# VPG baseline for comparison
!python -m spinup.run vpg_pytorch \
    --env LunarLander-v2 \
    --epochs 150 \
    --seed 0 10 20 \
    --exp_name vpg-lunar

In [ ]:
import glob
data_dirs = sorted(glob.glob('/root/spinningup/data/ppo-lunar*/') +
                   glob.glob('/root/spinningup/data/vpg-lunar*/'))
dir_str = ' '.join(data_dirs)
!python -m spinup.run plot {dir_str}

**Analysis questions:**
- How many epochs until PPO reaches mean return > 200 ("solved")?
- Does VPG converge on LunarLander within 150 epochs?
- Which `lam` value converges faster, and why does a lower λ add bias?

## Exercise 4: ExperimentGrid Sweep

Systematic hyperparameter search: 3 seeds × 3 architectures × 3 clip ratios × 2 λ values = **54 experiments**.

In [ ]:
import sys
sys.path.insert(0, '/content/spinningup')

from spinup.utils.run_utils import ExperimentGrid
from spinup.algos.pytorch.ppo.ppo import ppo

eg = ExperimentGrid(name='ppo-lunar-sweep')
eg.add('env_name',               'LunarLander-v2',             '',    True)
eg.add('seed',                   [0, 10, 20])
eg.add('epochs',                 100)
eg.add('ac_kwargs:hidden_sizes', [(32,32), (64,64), (128,128)], 'hid')
eg.add('clip_ratio',             [0.1, 0.2, 0.3],               'clip')
eg.add('lam',                    [0.95, 0.97],                   'lam')

eg.run(ppo, num_cpu=1)

In [ ]:
import glob
sweep_dirs = sorted(glob.glob('/root/spinningup/data/ppo-lunar-sweep*/'))
dir_str = ' '.join(sweep_dirs)
!python -m spinup.run plot {dir_str}

**Discussion:**
- Which architecture performed best on average?
- Is there an interaction between `clip_ratio` and `lam`?
- Which configuration has the lowest variance across seeds?

## Problem Set 1.1: Diagonal Gaussian Log-Likelihood

For a diagonal Gaussian with mean $\mu$ and standard deviation $\sigma$, the log-likelihood of a sample $x$ is:

$$\log \pi(x|\mu,\sigma) = -\frac{1}{2}\sum_i\left(\frac{(x_i-\mu_i)^2}{\sigma_i^2} + 2\log\sigma_i + \log 2\pi\right)$$

Implement your solution in `exercise1_1.py`, then run the auto-verification cell.

In [ ]:
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_1.py

In [ ]:
# ── YOUR IMPLEMENTATION ──────────────────────────────────────────
# Edit exercise1_1.py directly, or implement the function below
# and paste it in before running the verification.
#
# def gaussian_likelihood(x, mu, log_std):
#     """
#     Args:
#         x       : (batch, act_dim) sampled actions
#         mu      : (batch, act_dim) distribution means
#         log_std : (batch, act_dim) or (act_dim,) log standard deviations
#     Returns:
#         log_prob: (batch,) — sum of per-dimension log-likelihoods
#     """
#     # YOUR CODE HERE
#     pass
# ─────────────────────────────────────────────────────────────────

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_1.py

## Problem Set 1.2: MLP Diagonal Gaussian Policy for PPO

Implement an MLP diagonal Gaussian policy compatible with Spinning Up's PPO training loop.

Requirements:
- Returns a distribution with `.log_prob()` and `.sample()` using your Exercise 1.1 implementation
- Use a standalone `nn.Parameter` for log-std (not a network output) — more stable
- `log_prob` must return the **sum** across action dimensions (scalar per sample)

**Evaluation:** 20 epochs on `InvertedPendulum-v2`. Success = average score > 500 in the last 5 epochs.

In [ ]:
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_2.py

In [ ]:
# Implement MLPGaussianActor in exercise1_2.py, then evaluate:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_2.py

## Problem Set 1.3: TD3 Computation Graph

Implement the TD3 critic and policy loss functions. The starter code gives you everything except the loss computations — find `# YOUR CODE HERE`.

**Critic loss** (clipped double-Q with target smoothing):
$$y = r + \gamma(1-d)\min_{i=1,2}Q_{\phi_{\text{targ},i}}(s', \tilde{a}'), \quad \mathcal{L}(\phi)=\mathbb{E}[(Q_\phi(s,a)-y)^2]$$

**Policy loss** (delayed update, every `policy_delay` steps):
$$\mathcal{L}(\theta)=-\mathbb{E}[Q_{\phi_1}(s,\mu_\theta(s))]$$

**Evaluation:** Within 10 epochs — HalfCheetah-v2 > 300, InvertedPendulum-v2 = 150.

In [ ]:
!cat /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env InvertedPendulum-v2

In [ ]:
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env HalfCheetah-v2

In [ ]:
# Compare against the reference solution
!python /content/spinningup/spinup/exercises/pytorch/problem_set_1/exercise1_3.py \
    --env HalfCheetah-v2 --use_soln